In [ ]:
# ============================================================
# CELL 1
# IMPORTS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random
import time

import cv2
import numpy as np
import pandas as pd

from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import (
    mobilenet_v3_small,
    MobileNet_V3_Small_Weights
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.preprocessing import LabelEncoder

from numpy.linalg import inv

Mounted at /content/drive


In [ ]:
# ============================================================
# CELL 2
# DATASET SETUP
# ============================================================

DATASET_ROOT = "/content/drive/MyDrive/Cotton/"

SAVE_DIR = os.path.join(
    DATASET_ROOT,
    "run"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

RANDOM_STATE = 42

IMAGE_SIZE = 224

CLASS_NAMES = sorted([

    folder

    for folder in os.listdir(DATASET_ROOT)

    if os.path.isdir(
        os.path.join(DATASET_ROOT, folder)
    )

    and folder != "run"

])

print("="*60)
print("CLASSES")
print("="*60)

print(CLASS_NAMES)

# ------------------------------------------------------------
# Build dataframe
# ------------------------------------------------------------

rows = []

for cls in CLASS_NAMES:

    class_dir = os.path.join(
        DATASET_ROOT,
        cls
    )

    for file in os.listdir(class_dir):

        if file.lower().endswith((
            ".jpg",
            ".jpeg",
            ".png"
        )):

            rows.append({

                "Image_ID": file,

                "image_path":
                    os.path.join(class_dir, file),

                "class": cls

            })

df = pd.DataFrame(rows)

print()

print("Total Images :", len(df))

print()

print(df["class"].value_counts())

# ------------------------------------------------------------
# Encode labels
# ------------------------------------------------------------

label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(
    df["class"]
)

NUM_CLASSES = len(
    label_encoder.classes_
)

print()

print("Label Mapping")

for i, cls in enumerate(
    label_encoder.classes_
):

    print(i, "->", cls)

# ------------------------------------------------------------
# Train / Validation Split
# ------------------------------------------------------------

df_train, df_val = train_test_split(

    df,

    test_size=0.20,

    stratify=df["label"],

    random_state=RANDOM_STATE

)

df_train = df_train.reset_index(
    drop=True
)

df_val = df_val.reset_index(
    drop=True
)

print()

print("="*60)
print("TRAIN SPLIT")
print("="*60)

print(df_train["class"].value_counts())

print()

print("="*60)
print("VALIDATION SPLIT")
print("="*60)

print(df_val["class"].value_counts())

# ------------------------------------------------------------
# Save splits
# ------------------------------------------------------------

TRAIN_SPLIT_PATH = os.path.join(

    SAVE_DIR,

    "train_split.csv"

)

VAL_SPLIT_PATH = os.path.join(

    SAVE_DIR,

    "validation_split.csv"

)

df_train.to_csv(
    TRAIN_SPLIT_PATH,
    index=False
)

df_val.to_csv(
    VAL_SPLIT_PATH,
    index=False
)

print()

print("Saved")

print(TRAIN_SPLIT_PATH)

print(VAL_SPLIT_PATH)

# ============================================================
# LOAD PRETRAINED MOBILENET
# ============================================================

device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

weights = MobileNet_V3_Small_Weights.DEFAULT

model = mobilenet_v3_small(
    weights=weights
)

print()

print("="*60)
print("PRETRAINED MOBILENET LOADED")
print("="*60)

print(model)

# ============================================================
# CONVERT TO FEATURE EXTRACTOR
# ============================================================

feature_extractor = nn.Sequential(

    model.features,

    model.avgpool,

    nn.Flatten()

).to(device)

feature_extractor.eval()

dummy = torch.randn(
    1,
    3,
    224,
    224
).to(device)

with torch.no_grad():

    output = feature_extractor(dummy)

print()

print("="*60)
print("FEATURE EXTRACTOR")
print("="*60)

print("Feature Shape :", output.shape)

CLASSES
['Bacterial_Blight', 'Curl_Virus', 'Healthy_Leaf', 'Herbicide_Growth_Damage', 'Leaf_Hopper_Jassids', 'Leaf_Redding', 'Leaf_Variegation']

Total Images : 2137

class
Leaf_Redding               578
Curl_Virus                 431
Herbicide_Growth_Damage    280
Healthy_Leaf               257
Bacterial_Blight           250
Leaf_Hopper_Jassids        225
Leaf_Variegation           116
Name: count, dtype: int64

Label Mapping
0 -> Bacterial_Blight
1 -> Curl_Virus
2 -> Healthy_Leaf
3 -> Herbicide_Growth_Damage
4 -> Leaf_Hopper_Jassids
5 -> Leaf_Redding
6 -> Leaf_Variegation

TRAIN SPLIT
class
Leaf_Redding               462
Curl_Virus                 345
Herbicide_Growth_Damage    224
Healthy_Leaf               205
Bacterial_Blight           200
Leaf_Hopper_Jassids        180
Leaf_Variegation            93
Name: count, dtype: int64

VALIDATION SPLIT
class
Leaf_Redding               116
Curl_Virus                  86
Herbicide_Growth_Damage     56
Healthy_Leaf                52
Bacterial

100%|██████████| 9.83M/9.83M [00:00<00:00, 154MB/s]



PRETRAINED MOBILENET LOADED
MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d

In [ ]:
train_ids = set(df_train["Image_ID"])
val_ids = set(df_val["Image_ID"])

overlap = train_ids.intersection(val_ids)

print("Overlap:", len(overlap))

Overlap: 0


In [ ]:
# ============================================================
# SAVE CLASS NAMES
# ============================================================

CLASS_NAMES_PATH = os.path.join(
    SAVE_DIR,
    "class_names.json"
)

with open(CLASS_NAMES_PATH, "w") as f:
    json.dump(
        CLASS_NAMES,
        f,
        indent=2
    )

print("Class names saved to:")
print(CLASS_NAMES_PATH)

print()
print("Classes:")
print(CLASS_NAMES)

Class names saved to:
/content/drive/MyDrive/CottonDataset/run/class_names.json

Classes:
['Bacterial_Blight', 'Curl_Virus', 'Healthy_Leaf', 'Herbicide_Growth_Damage', 'Leaf_Hopper_Jassids', 'Leaf_Redding', 'Leaf_Variegation']


In [ ]:
# ============================================================
# CELL 3
# EXTRACT MOBILENET FEATURES
# ============================================================

TRAIN_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "train_features.csv"
)

VAL_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "validation_features.csv"
)

# ------------------------------------------------------------
# Image Transform
# ------------------------------------------------------------

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ------------------------------------------------------------
# Load Existing Features
# ------------------------------------------------------------

if (
    os.path.exists(TRAIN_FEATURE_PATH)
    and
    os.path.exists(VAL_FEATURE_PATH)
):

    print("=" * 60)
    print("FEATURE FILES FOUND")
    print("=" * 60)

    extracted_features_train = pd.read_csv(
        TRAIN_FEATURE_PATH
    )

    extracted_features_val = pd.read_csv(
        VAL_FEATURE_PATH
    )

    print(
        extracted_features_train.shape
    )

    print(
        extracted_features_val.shape
    )

else:

    # --------------------------------------------------------
    # Feature Extraction Function
    # --------------------------------------------------------

    def extract_features(dataframe):

        rows = []

        feature_extractor.eval()

        with torch.no_grad():

            for _, row in tqdm(
                dataframe.iterrows(),
                total=len(dataframe)
            ):

                image = Image.open(
                    row["image_path"]
                ).convert("RGB")

                image = transform(image)

                image = image.unsqueeze(0).to(device)

                features = feature_extractor(
                    image
                )

                features = (
                    features
                    .cpu()
                    .numpy()
                    .flatten()
                )

                rows.append([

                    row["Image_ID"],

                    row["class"],

                    *features

                ])

        columns = (

            ["Image_ID", "class"]

            +

            [
                f"feat_{i}"
                for i in range(576)
            ]

        )

        return pd.DataFrame(
            rows,
            columns=columns
        )

    # --------------------------------------------------------
    # Training Features
    # --------------------------------------------------------

    print()

    print("=" * 60)
    print("TRAIN FEATURES")
    print("=" * 60)

    extracted_features_train = extract_features(
        df_train
    )

    # --------------------------------------------------------
    # Validation Features
    # --------------------------------------------------------

    print()

    print("=" * 60)
    print("VALIDATION FEATURES")
    print("=" * 60)

    extracted_features_val = extract_features(
        df_val
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    extracted_features_train.to_csv(

        TRAIN_FEATURE_PATH,

        index=False

    )

    extracted_features_val.to_csv(

        VAL_FEATURE_PATH,

        index=False

    )

    print()

    print("Saved")

    print(TRAIN_FEATURE_PATH)

    print(VAL_FEATURE_PATH)

print()

print(extracted_features_train.head())

print()

print(extracted_features_train.shape)

print(extracted_features_val.shape)


TRAIN FEATURES


100%|██████████| 1709/1709 [10:11<00:00,  2.80it/s]



VALIDATION FEATURES


100%|██████████| 428/428 [02:29<00:00,  2.87it/s]



Saved
/content/drive/MyDrive/CottonDataset/run/train_features.csv
/content/drive/MyDrive/CottonDataset/run/validation_features.csv

      Image_ID         class    feat_0    feat_1    feat_2    feat_3  \
0  LR00415.jpg  Leaf_Redding  0.022164 -0.026829  0.059013  0.999744   
1  CV00228.jpg    Curl_Virus -0.007485 -0.106434  0.146866  0.079160   
2  CV00039.jpg    Curl_Virus -0.085287 -0.106937 -0.151679  0.036730   
3  HL00050.jpg  Healthy_Leaf -0.202001 -0.024298 -0.115852 -0.098176   
4  HL00249.jpg  Healthy_Leaf -0.139253 -0.080446 -0.142856 -0.162585   

     feat_4    feat_5    feat_6    feat_7  ...  feat_566  feat_567  feat_568  \
0  0.888762  0.191093  0.068747  0.067164  ...  0.880877  1.257823  0.001794   
1  0.742036  0.764833 -0.009958 -0.120061  ... -0.026285  0.457067 -0.079265   
2  0.960491 -0.125345 -0.107312  0.351930  ...  0.019166  1.693027  0.433996   
3  1.981039 -0.127559 -0.124132  0.275587  ... -0.082691  1.121294  0.462119   
4  0.711251 -0.139575 -0.128599  0

In [ ]:
# ============================================================
# CELL 4
# TRAIN JACOBI-DMR
# ============================================================

import time

time_jacobi_start = time.time()

# ------------------------------------------------------------
# Prepare Training Data
# ------------------------------------------------------------

X_train = extracted_features_train.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_train = extracted_features_train["class"]

X_train = X_train.values

# ------------------------------------------------------------
# One-Hot Encoding
# ------------------------------------------------------------

y_one_hot = pd.get_dummies(
    y_train
)

# ------------------------------------------------------------
# Jacobi Hyperparameters
# ------------------------------------------------------------

a = b = 1 / len(X_train)

k = 1

# ------------------------------------------------------------
# Train One Classifier Per Class
# ------------------------------------------------------------

betas = {}

for cls in y_one_hot.columns:

    eta = np.log(

        (y_one_hot[cls] + a)

        /

        (1 + k * b)

    )

    beta = inv(

        X_train.T @ X_train

    ) @ (

        X_train.T @ eta

    )

    betas[cls] = beta

# ------------------------------------------------------------
# Prepare Validation Data
# ------------------------------------------------------------

X_val = extracted_features_val.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_true = extracted_features_val["class"]

X_val = X_val.values

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

lambda_scores = {}

for cls in betas:

    lambda_scores[cls] = np.exp(
        X_val @ betas[cls]
    )

lambda_df = pd.DataFrame(
    lambda_scores
)

y_pred = lambda_df.idxmax(
    axis=1
)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

time_jacobi_seconds = (
    time.time() -
    time_jacobi_start
)

acc_jacobi = accuracy_score(
    y_true,
    y_pred
)

cm_jacobi = confusion_matrix(
    y_true,
    y_pred,
    labels=CLASS_NAMES
)

print()

print("=" * 60)
print("JACOBI-DMR")
print("=" * 60)

print(
    f"Accuracy: {acc_jacobi:.4f}"
)

print(
    f"Time: {time_jacobi_seconds:.4f}s"
)

print()

print(
    classification_report(
        y_true,
        y_pred,
        labels=CLASS_NAMES
    )
)

print()

print("Confusion Matrix:\n")

print(cm_jacobi)

# ------------------------------------------------------------
# Save Coefficients
# ------------------------------------------------------------

JACOBI_PATH = os.path.join(
    SAVE_DIR,
    "jacobi_coefficients.json"
)

with open(
    JACOBI_PATH,
    "w"
) as f:

    json.dump(
        {
            cls: betas[cls].tolist()
            for cls in betas
        },
        f
    )

print()

print("Jacobi coefficients saved to:")

print(JACOBI_PATH)


JACOBI-DMR
Accuracy: 0.9439
Time: 0.1888s

                         precision    recall  f1-score   support

       Bacterial_Blight       0.84      0.96      0.90        50
             Curl_Virus       0.94      0.94      0.94        86
           Healthy_Leaf       0.96      0.96      0.96        52
Herbicide_Growth_Damage       0.98      0.96      0.97        56
    Leaf_Hopper_Jassids       0.90      0.96      0.92        45
           Leaf_Redding       0.98      0.91      0.94       116
       Leaf_Variegation       1.00      1.00      1.00        23

               accuracy                           0.94       428
              macro avg       0.94      0.96      0.95       428
           weighted avg       0.95      0.94      0.94       428


Confusion Matrix:

[[ 48   0   1   0   0   1   0]
 [  1  81   0   1   2   1   0]
 [  0   1  50   0   1   0   0]
 [  0   2   0  54   0   0   0]
 [  1   1   0   0  43   0   0]
 [  7   1   1   0   2 105   0]
 [  0   0   0   0   0   0  23]]


In [ ]:
# ============================================================
# RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestClassifier

import time

time_rf_start = time.time()

# ------------------------------------------------------------
# Training Data
# ------------------------------------------------------------

X_train = extracted_features_train.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

Y_train = extracted_features_train["class"]

# ------------------------------------------------------------
# Validation Data
# ------------------------------------------------------------

X_val = extracted_features_val.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_true = extracted_features_val["class"]

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(
    X_train,
    Y_train
)

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

y_pred_rf = rf_model.predict(
    X_val
)

time_rf_seconds = (
    time.time() -
    time_rf_start
)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

acc_rf = accuracy_score(
    y_true,
    y_pred_rf
)

cm_rf = confusion_matrix(
    y_true,
    y_pred_rf,
    labels=CLASS_NAMES
)

print()

print("=" * 60)
print("RANDOM FOREST")
print("=" * 60)

print(
    f"Accuracy: {acc_rf:.4f}"
)

print(
    f"Time: {time_rf_seconds:.4f}s"
)

print()

print(
    classification_report(
        y_true,
        y_pred_rf,
        labels=CLASS_NAMES
    )
)

print()

print("Confusion Matrix:\n")

print(cm_rf)


RANDOM FOREST
Accuracy: 0.8995
Time: 3.0555s

                         precision    recall  f1-score   support

       Bacterial_Blight       0.87      0.82      0.85        50
             Curl_Virus       0.85      0.95      0.90        86
           Healthy_Leaf       0.88      0.96      0.92        52
Herbicide_Growth_Damage       0.96      0.96      0.96        56
    Leaf_Hopper_Jassids       0.89      0.71      0.79        45
           Leaf_Redding       0.92      0.91      0.92       116
       Leaf_Variegation       0.95      0.87      0.91        23

               accuracy                           0.90       428
              macro avg       0.90      0.88      0.89       428
           weighted avg       0.90      0.90      0.90       428


Confusion Matrix:

[[ 41   2   3   1   0   3   0]
 [  0  82   1   1   1   1   0]
 [  0   2  50   0   0   0   0]
 [  0   2   0  54   0   0   0]
 [  2   4   2   0  32   4   1]
 [  4   4   0   0   2 106   0]
 [  0   0   1   0   1   1  20

In [ ]:
# ============================================================
# SUPPORT VECTOR MACHINE (SVM)
# ============================================================

from sklearn.svm import SVC

import time

time_svm_start = time.time()

# ------------------------------------------------------------
# Training Data
# ------------------------------------------------------------

X_train = extracted_features_train.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

Y_train = extracted_features_train["class"]

# ------------------------------------------------------------
# Validation Data
# ------------------------------------------------------------

X_val = extracted_features_val.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_true = extracted_features_val["class"]

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    random_state=42
)

svm_model.fit(
    X_train,
    Y_train
)

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

y_pred_svm = svm_model.predict(
    X_val
)

time_svm_seconds = (
    time.time() -
    time_svm_start
)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

acc_svm = accuracy_score(
    y_true,
    y_pred_svm
)

cm_svm = confusion_matrix(
    y_true,
    y_pred_svm,
    labels=CLASS_NAMES
)

print()

print("=" * 60)
print("SUPPORT VECTOR MACHINE")
print("=" * 60)

print(
    f"Accuracy: {acc_svm:.4f}"
)

print(
    f"Time: {time_svm_seconds:.4f}s"
)

print()

print(
    classification_report(
        y_true,
        y_pred_svm,
        labels=CLASS_NAMES
    )
)

print()

print("Confusion Matrix:\n")

print(cm_svm)


SUPPORT VECTOR MACHINE
Accuracy: 0.9626
Time: 0.3548s

                         precision    recall  f1-score   support

       Bacterial_Blight       0.91      0.96      0.93        50
             Curl_Virus       0.96      1.00      0.98        86
           Healthy_Leaf       0.96      1.00      0.98        52
Herbicide_Growth_Damage       1.00      0.98      0.99        56
    Leaf_Hopper_Jassids       0.91      0.93      0.92        45
           Leaf_Redding       0.99      0.91      0.95       116
       Leaf_Variegation       1.00      1.00      1.00        23

               accuracy                           0.96       428
              macro avg       0.96      0.97      0.97       428
           weighted avg       0.96      0.96      0.96       428


Confusion Matrix:

[[ 48   1   1   0   0   0   0]
 [  0  86   0   0   0   0   0]
 [  0   0  52   0   0   0   0]
 [  0   1   0  55   0   0   0]
 [  1   0   1   0  42   1   0]
 [  4   2   0   0   4 106   0]
 [  0   0   0   0   

In [ ]:
# ============================================================
# RIDGE CLASSIFIER
# ============================================================

from sklearn.linear_model import RidgeClassifier

import time

time_ridge_start = time.time()

# ------------------------------------------------------------
# Training Data
# ------------------------------------------------------------

X_train = extracted_features_train.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

Y_train = extracted_features_train["class"]

# ------------------------------------------------------------
# Validation Data
# ------------------------------------------------------------

X_val = extracted_features_val.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_true = extracted_features_val["class"]

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

ridge_model = RidgeClassifier(
    alpha=1.0,
    random_state=42
)

ridge_model.fit(
    X_train,
    Y_train
)

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

y_pred_ridge = ridge_model.predict(
    X_val
)

time_ridge_seconds = (
    time.time() -
    time_ridge_start
)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

acc_ridge = accuracy_score(
    y_true,
    y_pred_ridge
)

cm_ridge = confusion_matrix(
    y_true,
    y_pred_ridge,
    labels=CLASS_NAMES
)

print()

print("=" * 60)
print("RIDGE CLASSIFIER")
print("=" * 60)

print(
    f"Accuracy: {acc_ridge:.4f}"
)

print(
    f"Time: {time_ridge_seconds:.4f}s"
)

print()

print(
    classification_report(
        y_true,
        y_pred_ridge,
        labels=CLASS_NAMES
    )
)

print()

print("Confusion Matrix:\n")

print(cm_ridge)


RIDGE CLASSIFIER
Accuracy: 0.9509
Time: 0.0702s

                         precision    recall  f1-score   support

       Bacterial_Blight       0.86      0.96      0.91        50
             Curl_Virus       0.95      0.97      0.96        86
           Healthy_Leaf       0.96      0.96      0.96        52
Herbicide_Growth_Damage       1.00      0.98      0.99        56
    Leaf_Hopper_Jassids       0.91      0.93      0.92        45
           Leaf_Redding       0.98      0.91      0.95       116
       Leaf_Variegation       0.96      1.00      0.98        23

               accuracy                           0.95       428
              macro avg       0.95      0.96      0.95       428
           weighted avg       0.95      0.95      0.95       428


Confusion Matrix:

[[ 48   0   1   0   0   1   0]
 [  1  83   0   0   1   1   0]
 [  0   1  50   0   1   0   0]
 [  0   1   0  55   0   0   0]
 [  1   1   0   0  42   0   1]
 [  6   1   1   0   2 106   0]
 [  0   0   0   0   0   0 

In [ ]:
# ============================================================
# LOGISTIC REGRESSION (L1 / LASSO)
# ============================================================

from sklearn.linear_model import LogisticRegression

import time

time_lasso_start = time.time()

# ------------------------------------------------------------
# Training Data
# ------------------------------------------------------------

X_train = extracted_features_train.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

Y_train = extracted_features_train["class"]

# ------------------------------------------------------------
# Validation Data
# ------------------------------------------------------------

X_val = extracted_features_val.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_true = extracted_features_val["class"]

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

lasso_model = LogisticRegression(
    penalty="l1",
    solver="saga",
    C=1.0,
    max_iter=5000,
    random_state=42
)

lasso_model.fit(
    X_train,
    Y_train
)

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

y_pred_lasso = lasso_model.predict(
    X_val
)

time_lasso_seconds = (
    time.time() -
    time_lasso_start
)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

acc_lasso = accuracy_score(
    y_true,
    y_pred_lasso
)

cm_lasso = confusion_matrix(
    y_true,
    y_pred_lasso,
    labels=CLASS_NAMES
)

print()

print("=" * 60)
print("LOGISTIC REGRESSION (L1 / LASSO)")
print("=" * 60)

print(
    f"Accuracy: {acc_lasso:.4f}"
)

print(
    f"Time: {time_lasso_seconds:.4f}s"
)

print()

print(
    classification_report(
        y_true,
        y_pred_lasso,
        labels=CLASS_NAMES
    )
)

print()

print("Confusion Matrix:\n")

print(cm_lasso)


LOGISTIC REGRESSION (L1 / LASSO)
Accuracy: 0.9369
Time: 70.2525s

                         precision    recall  f1-score   support

       Bacterial_Blight       0.90      0.92      0.91        50
             Curl_Virus       0.90      0.94      0.92        86
           Healthy_Leaf       1.00      0.96      0.98        52
Herbicide_Growth_Damage       1.00      0.95      0.97        56
    Leaf_Hopper_Jassids       0.85      0.91      0.88        45
           Leaf_Redding       0.96      0.92      0.94       116
       Leaf_Variegation       0.96      1.00      0.98        23

               accuracy                           0.94       428
              macro avg       0.94      0.94      0.94       428
           weighted avg       0.94      0.94      0.94       428


Confusion Matrix:

[[ 46   1   0   0   2   1   0]
 [  0  81   0   0   3   2   0]
 [  0   2  50   0   0   0   0]
 [  1   2   0  53   0   0   0]
 [  0   2   0   0  41   2   0]
 [  4   2   0   0   2 107   1]
 [  0   0

In [ ]:
# ============================================================
# FINAL MODEL COMPARISON
# ============================================================

comparison_df = pd.DataFrame({

    "Model": [

        "Jacobi-DMR",
        "Random Forest",
        "Support Vector Machine",
        "Ridge Classifier",
        "Logistic Regression (L1)"

    ],

    "Accuracy": [

        acc_jacobi,
        acc_rf,
        acc_svm,
        acc_ridge,
        acc_lasso

    ],

    "Execution Time (s)": [

        time_jacobi_seconds,
        time_rf_seconds,
        time_svm_seconds,
        time_ridge_seconds,
        time_lasso_seconds

    ]

})

comparison_df["Accuracy"] = comparison_df["Accuracy"].round(4)

comparison_df["Execution Time (s)"] = comparison_df[
    "Execution Time (s)"
].round(4)

comparison_df = comparison_df.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

print()

print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

print(comparison_df)

comparison_path = os.path.join(
    SAVE_DIR,
    "model_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print()

print(f"Comparison saved to:\n{comparison_path}")


MODEL COMPARISON
                      Model  Accuracy  Execution Time (s)
0    Support Vector Machine    0.9626              0.3548
1          Ridge Classifier    0.9509              0.0702
2                Jacobi-DMR    0.9439              0.1888
3  Logistic Regression (L1)    0.9369             70.2525
4             Random Forest    0.8995              3.0555

Comparison saved to:
/content/drive/MyDrive/CottonDataset/run/model_comparison.csv
